In [ ]:
import os
import json
import pickle
import time
import random
import signal
from json.decoder import NaN

import pandas as pd
import itertools
import sys
sys.path.insert(0, "..")
from Algorithms.LExaBan.BanzhafCircuit import DNFCircuit as LExaBan
from Algorithms.LExaShap.ShapleyCircuit import DNFCircuit as LExaShap
from Algorithms.Max_LExaBan.ArithmeticCircuit import ArithmeticCircuit


In [ ]:
def handler(signum, frame):
    raise TimeoutError("Timed out!")

signal.signal(signal.SIGALRM, handler)
signal.alarm(3)
try:
    time.sleep(10)   # pure Python-level blocking call
except TimeoutError:
    print("Interrupted!")
finally:
    signal.alarm(0)

In [ ]:
def parse_lineage(lineage):
    """
    Parses a lineage string into a better working format.
    """
    numbered_mapping = dict()
    vars = set()
    for clause in lineage:
        vars.update(clause)
    for idx, var in enumerate(vars):
        numbered_mapping[var] = idx
    res = [set([numbered_mapping[fact] for fact in clause]) for clause in lineage]

    return res

In [ ]:
sys.set_int_max_str_digits(0)

def handler(signum, frame):
    raise TimeoutError("Timed out!")

TIMEOUT_IN_S = 600

base_dir = os.path.join("data", 'lineage')
schemas = ["application", "offer", "workflow"]
granularity = ["context", "event", "relation"]

def compute_dnf_lineage_based_value(lineage, parsed_lineage, name:str = "LExaBan"):
    values = banzhaf_values if name == "LExaBan" else shapley_values

    try:
        c = LExaBan(parsed_lineage) if name == "LExaBan" else LExaShap(parsed_lineage)
        vals = c.banzhaf_values if name == "LExaBan" else c.shapley_values
        values.append((lineage[0], lineage[1], lineage[2], name, vals))
    except Exception as e:
        values.append((lineage[0], lineage[1], lineage[2], name, None))
        raise e # cancel the alarm

def load_lineage(base_dir, schema, gran, file_name):
    lineages = []
    file_path = os.path.join(base_dir, schema, gran, file_name)

    if not os.path.exists(file_path):
        return None

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        for lin in data:
            lineages.append((file_path[:-5], lin.get("src_activity", ""), lin.get("dest_activity", ""), lin.get("formula", "")))
    print(len(lineages))
    return lineages

def compute_dnf_lineage_based_values(lineages, algo, schema, gran, file_name):
    signal.signal(signal.SIGALRM, handler)
    signal.alarm(TIMEOUT_IN_S)  # seconds
    try:
        start = time.time()
        for i, lineage in enumerate(lineages):
            parsed_lineage = parse_lineage(lineage[3])
            compute_dnf_lineage_based_value(lineage, parsed_lineage, algo)

            if i % 100 == 0:
                print(f"Processed {i} lineages")
        end = time.time()
        runtimes.append(("Compute values", schema, gran, file_name[:-5], algo, end-start))
    except TimeoutError as e:
        print(e)
        runtimes.append(("Compute values", schema, gran, file_name[:-5], algo, None))
    except Exception as e:
        print(e)
        runtimes.append(("Compute values", schema, gran, file_name[:-5], algo, None))
    finally:
        signal.alarm(0)


for k in range(3):
    file_names = ["simple_handover_lineage.json", "alternate_response_viol_lineage.json", "efg_lineage.json"]
    runtimes = []
    banzhaf_values = []
    shapley_values = []
    # Computation for DNF-based queries
    for schema, gran, file_name in itertools.product(schemas, granularity, file_names):
        print("Start run for", schema, gran, file_name)
        loaded_lineages = load_lineage(base_dir, schema, gran, file_name)

        if loaded_lineages is None:
            continue

        compute_dnf_lineage_based_values(loaded_lineages, "LExaBan", schema, gran, file_name)
        compute_dnf_lineage_based_values(loaded_lineages, "LExaShap", schema, gran, file_name)

        print("Finished run for", schema, gran, file_name)

    df = pd.DataFrame(runtimes, columns=["step", "schema", "granularity", "lineage_type", "algorithm", "duration_seconds"])

    df.to_csv(os.path.join("Example Data", 'lineage_new', f'time_eval_results_dnf_{k}.csv'), index=False)

    if k < 1:
        with open(os.path.join("Example Data", 'lineage_new', 'shapley_values_eval_dnf.pkl'), 'wb') as f:
            pickle.dump(shapley_values, f)

        with open(os.path.join("Example Data", 'lineage_new', 'banzhaf_values_eval_dnf.pkl'), 'wb') as f:
            pickle.dump(banzhaf_values, f)

    # Computation for frequency based queries
    file_names = ["freq_handover_lineage.json", "freq_efg_lineage.json"]

    for schema, gran, file_name in itertools.product(schemas, granularity, file_names):
        file_path = os.path.join(base_dir, schema, gran, file_name)

        if not os.path.exists(file_path):
            continue
        elif schema == "workflow" and file_name == "freq_efg_lineage.json":
            runtimes.append(("Compute values", schema, gran, file_name[:-5], 'LExaBan', None))

        lineages = []
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            for lin in data:
                new_lin = [(list(set(clause)), val) for clause, val in lin.get("formula", "")]
                lineages.append((file_path[:-5], lin.get("src_activity", ""), lin.get("dest_activity", ""), new_lin))

        signal.signal(signal.SIGALRM, handler)
        signal.alarm(TIMEOUT_IN_S)
        try:
            start = time.time()
            for i, r in enumerate(lineages):
                try:
                    semimodule = r[3]
                    circ = ArithmeticCircuit(semimodule)
                    banzhaf_values.append((r[0], r[1], r[2], 'LExaBan', circ.banzhaf_values))
                except Exception as e:
                    print(e)
                    banzhaf_values.append((r[0], r[1], r[2], 'LExaBan', None))
            end = time.time()
            runtimes.append(("Compute values", schema, gran, file_name[:-5], 'LExaBan', end-start))
        except TimeoutError as e:
            print(e, file_path)
            runtimes.append(("Compute values", schema, gran, file_name[:-5], 'LExaBan', None))
        finally:
            signal.alarm(0)

    df = pd.DataFrame(runtimes, columns=["step", "schema", "granularity", "lineage_type", "algorithm", "duration_seconds"])

    df.to_csv(os.path.join("Example Data", 'lineage_new', f'time_eval_results{k}.csv'), index=False)

    if k < 1:
        with open(os.path.join("Example Data", 'lineage_new', 'shapley_values_eval.pkl'), 'wb') as f:
            pickle.dump(shapley_values, f)

        with open(os.path.join("Example Data", 'lineage_new', 'banzhaf_values_eval.pkl'), 'wb') as f:
            pickle.dump(banzhaf_values, f)